In [1]:
import requests
import pandas as pd

locations = {
    "Houston": (29.76, -95.37),
    "Austin": (30.27, -97.74),
    "San Antonio": (29.42, -98.49),
    "El Paso": (31.76, -106.49),
    "Abilene": (32.45, -99.73),
    "Tyler": (32.35, -95.30),
    "Wichita Falls": (33.91, -98.49)
}

print("Locations configured:", len(locations))

Locations configured: 7


In [2]:
url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": 29.76,
    "longitude": -95.37,
    "start_date": "2025-01-01",
    "end_date": "2025-12-31",
    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "apparent_temperature"
    ],
    "timezone": "GMT",
    "models": "era5"
}

response = requests.get(url, params=params, timeout=60)
response.raise_for_status()

data = response.json()

print("Hourly records:", len(data["hourly"]["time"]))
print("Variables:", list(data["hourly"].keys()))

Hourly records: 8760
Variables: ['time', 'temperature_2m', 'relative_humidity_2m', 'apparent_temperature']


In [3]:
from pathlib import Path

weather_frames = []

for city, (lat, lon) in locations.items():

    params["latitude"] = lat
    params["longitude"] = lon

    response = requests.get(
        url,
        params=params,
        timeout=60
    )
    response.raise_for_status()

    city_data = pd.DataFrame(
        response.json()["hourly"]
    )

    city_data["city"] = city
    weather_frames.append(city_data)

    print(city, ":", len(city_data), "records")

Houston : 8760 records
Austin : 8760 records
San Antonio : 8760 records
El Paso : 8760 records
Abilene : 8760 records
Tyler : 8760 records
Wichita Falls : 8760 records


In [4]:
all_weather = pd.concat(
    weather_frames,
    ignore_index=True
)

all_weather["hour_utc"] = pd.to_datetime(
    all_weather["time"],
    utc=True
)

print("Total rows:", len(all_weather))
print("Cities:", all_weather["city"].nunique())

print("\nMissing values:")
print(all_weather.isnull().sum())

print("\nDuplicate city-hour records:")
print(
    all_weather.duplicated(
        subset=["city", "hour_utc"]
    ).sum()
)

Total rows: 61320
Cities: 7

Missing values:
time                    0
temperature_2m          0
relative_humidity_2m    0
apparent_temperature    0
city                    0
hour_utc                0
dtype: int64

Duplicate city-hour records:
0


In [5]:
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

all_weather.to_csv(
    output_dir / "texas_weather_7cities_2025.csv",
    index=False
)

print("Weather dataset exported successfully!")
print("Shape:", all_weather.shape)

Weather dataset exported successfully!
Shape: (61320, 6)


In [6]:
dallas = pd.read_csv(
    "../data/processed/dallas_weather_clean_2025.csv"
)

dallas = dallas.rename(columns={
    "temperature_c": "temperature_2m",
    "humidity_pct": "relative_humidity_2m",
    "feels_like_c": "apparent_temperature"
})

dallas = dallas[
    ["time", "temperature_2m",
     "relative_humidity_2m",
     "apparent_temperature", "city", "hour_utc"]
]

print("Dallas records:", len(dallas))
print(dallas.head(3))

Dallas records: 8760
               time  temperature_2m  relative_humidity_2m  \
0  2025-01-01T00:00            10.2                    60   
1  2025-01-01T01:00            10.0                    62   
2  2025-01-01T02:00             8.4                    69   

   apparent_temperature    city                   hour_utc  
0                   7.3  Dallas  2025-01-01 00:00:00+00:00  
1                   7.2  Dallas  2025-01-01 01:00:00+00:00  
2                   5.6  Dallas  2025-01-01 02:00:00+00:00  


In [7]:
weather_8cities = pd.concat(
    [all_weather, dallas],
    ignore_index=True
)

weather_8cities["hour_utc"] = pd.to_datetime(
    weather_8cities["hour_utc"],
    utc=True
)

print("Total records:", len(weather_8cities))
print("Cities:", weather_8cities["city"].nunique())

print("Missing values:")
print(weather_8cities.isnull().sum())

print("Duplicate city-hour records:")
print(
    weather_8cities.duplicated(
        subset=["city", "hour_utc"]
    ).sum()
)

Total records: 70080
Cities: 8
Missing values:
time                    0
temperature_2m          0
relative_humidity_2m    0
apparent_temperature    0
city                    0
hour_utc                0
dtype: int64
Duplicate city-hour records:
0


In [8]:
weather_8cities.to_csv(
    output_dir / "texas_weather_8cities_2025.csv",
    index=False
)

print("Final weather dataset exported!")
print("Shape:", weather_8cities.shape)

Final weather dataset exported!
Shape: (70080, 6)


In [9]:
dallas_params = params.copy()

dallas_params["latitude"] = 32.78
dallas_params["longitude"] = -96.80
dallas_params["models"] = "era5"

response = requests.get(
    url, params=dallas_params, timeout=60
)
response.raise_for_status()

dallas_era5 = pd.DataFrame(
    response.json()["hourly"]
)

dallas_era5["city"] = "Dallas"
dallas_era5["hour_utc"] = pd.to_datetime(
    dallas_era5["time"], utc=True
)

print("Dallas ERA5 records:", len(dallas_era5))

Dallas ERA5 records: 8760


In [10]:
weather_final = pd.concat(
    [all_weather, dallas_era5],
    ignore_index=True
)

weather_final["hour_utc"] = pd.to_datetime(
    weather_final["hour_utc"], utc=True
)

print("Shape:", weather_final.shape)
print("Cities:", weather_final["city"].nunique())
print("Missing values:", weather_final.isna().sum().sum())
print(
    "Duplicate city-hour records:",
    weather_final.duplicated(
        ["city", "hour_utc"]
    ).sum()
)

Shape: (70080, 6)
Cities: 8
Missing values: 0
Duplicate city-hour records: 0


In [11]:
weather_final.to_csv(
    output_dir / "texas_weather_era5_2025.csv",
    index=False
)

print("Consistent ERA5 weather dataset exported!")

Consistent ERA5 weather dataset exported!


In [12]:
import pandas as pd

electricity = pd.read_csv(
    "../data/processed/ercot_statewide_2025.csv"
)

weather = pd.read_csv(
    "../data/processed/texas_weather_era5_2025.csv"
)

electricity["hour_ending_utc"] = pd.to_datetime(
    electricity["hour_ending_utc"], utc=True
)

weather["hour_utc"] = pd.to_datetime(
    weather["hour_utc"], utc=True
)

print("Electricity:", electricity.shape)
print("Weather:", weather.shape)
print("Cities:", weather["city"].unique())

Electricity: (8760, 6)
Weather: (70080, 6)
Cities: <StringArray>
[      'Houston',        'Austin',   'San Antonio',       'El Paso',
       'Abilene',         'Tyler', 'Wichita Falls',        'Dallas']
Length: 8, dtype: str


In [13]:
weather_statewide = (
    weather.groupby("hour_utc")
    .agg(
        avg_temperature_c=("temperature_2m", "mean"),
        avg_humidity_pct=("relative_humidity_2m", "mean"),
        avg_feels_like_c=("apparent_temperature", "mean")
    )
    .reset_index()
)

print("Weather summary:", weather_statewide.shape)
print(weather_statewide.head())

Weather summary: (8760, 4)
                   hour_utc  avg_temperature_c  avg_humidity_pct  \
0 2025-01-01 00:00:00+00:00            12.8375            46.000   
1 2025-01-01 01:00:00+00:00            12.6250            46.625   
2 2025-01-01 02:00:00+00:00            10.5250            53.000   
3 2025-01-01 03:00:00+00:00             9.8875            55.250   
4 2025-01-01 04:00:00+00:00             9.1125            57.250   

   avg_feels_like_c  
0            9.5625  
1            9.2750  
2            7.0750  
3            6.4125  
4            5.5500  


In [14]:
electricity["weather_hour_utc"] = (
    electricity["hour_ending_utc"]
    - pd.Timedelta(hours=1)
)

integrated = electricity.merge(
    weather_statewide,
    left_on="weather_hour_utc",
    right_on="hour_utc",
    how="left",
    validate="one_to_one"
)

print("Integrated shape:", integrated.shape)

print("Matched records:",
      integrated["avg_temperature_c"].notna().sum())

print("Unmatched records:",
      integrated["avg_temperature_c"].isna().sum())

print(integrated.head(3))

Integrated shape: (8760, 11)
Matched records: 8754
Unmatched records: 6
            hour_ending_utc        date  month day_of_week  hour  \
0 2025-01-01 07:00:00+00:00  2025-01-01      1   Wednesday     1   
1 2025-01-01 08:00:00+00:00  2025-01-01      1   Wednesday     2   
2 2025-01-01 09:00:00+00:00  2025-01-01      1   Wednesday     3   

      demand_mw          weather_hour_utc                  hour_utc  \
0  43223.859905 2025-01-01 06:00:00+00:00 2025-01-01 06:00:00+00:00   
1  43122.532212 2025-01-01 07:00:00+00:00 2025-01-01 07:00:00+00:00   
2  42918.511847 2025-01-01 08:00:00+00:00 2025-01-01 08:00:00+00:00   

   avg_temperature_c  avg_humidity_pct  avg_feels_like_c  
0             7.7750            62.625            4.2125  
1             7.3875            63.750            3.7750  
2             6.0875            67.875            2.3625  


In [15]:
# Find electricity hours without matching weather
unmatched = integrated[
    integrated["avg_temperature_c"].isna()
]

print("Unmatched records:", len(unmatched))

print(
    unmatched[
        ["hour_ending_utc", "weather_hour_utc"]
    ].to_string(index=False)
)

Unmatched records: 6
          hour_ending_utc          weather_hour_utc
2026-01-01 01:00:00+00:00 2026-01-01 00:00:00+00:00
2026-01-01 02:00:00+00:00 2026-01-01 01:00:00+00:00
2026-01-01 03:00:00+00:00 2026-01-01 02:00:00+00:00
2026-01-01 04:00:00+00:00 2026-01-01 03:00:00+00:00
2026-01-01 05:00:00+00:00 2026-01-01 04:00:00+00:00
2026-01-01 06:00:00+00:00 2026-01-01 05:00:00+00:00


In [16]:
print("Electricity UTC range:")
print(electricity["hour_ending_utc"].min())
print(electricity["hour_ending_utc"].max())

print("\nWeather UTC range:")
print(weather["hour_utc"].min())
print(weather["hour_utc"].max())

Electricity UTC range:
2025-01-01 07:00:00+00:00
2026-01-01 06:00:00+00:00

Weather UTC range:
2025-01-01 00:00:00+00:00
2025-12-31 23:00:00+00:00


In [17]:
unmatched = integrated[
    integrated["avg_temperature_c"].isna()
]

print("Unmatched records:", len(unmatched))

print(
    unmatched[
        ["hour_ending_utc", "weather_hour_utc"]
    ].to_string(index=False)
)

Unmatched records: 6
          hour_ending_utc          weather_hour_utc
2026-01-01 01:00:00+00:00 2026-01-01 00:00:00+00:00
2026-01-01 02:00:00+00:00 2026-01-01 01:00:00+00:00
2026-01-01 03:00:00+00:00 2026-01-01 02:00:00+00:00
2026-01-01 04:00:00+00:00 2026-01-01 03:00:00+00:00
2026-01-01 05:00:00+00:00 2026-01-01 04:00:00+00:00
2026-01-01 06:00:00+00:00 2026-01-01 05:00:00+00:00


In [18]:
import requests
import pandas as pd

cities = {
    "Houston": (29.7604, -95.3698),
    "Austin": (30.2672, -97.7431),
    "San Antonio": (29.4241, -98.4936),
    "El Paso": (31.7619, -106.4850),
    "Abilene": (32.4487, -99.7331),
    "Tyler": (32.3513, -95.3011),
    "Wichita Falls": (33.9137, -98.4934),
    "Dallas": (32.7767, -96.7970)
}

extra_weather = []

for city, (lat, lon) in cities.items():

    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": "2026-01-01",
        "end_date": "2026-01-01",
        "hourly": (
            "temperature_2m,"
            "relative_humidity_2m,"
            "apparent_temperature"
        ),
        "timezone": "UTC",
        "models": "era5"
    }

    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()
    data = response.json()["hourly"]

    df = pd.DataFrame(data)

    df["hour_utc"] = pd.to_datetime(
        df["time"], utc=True
    )

    df = df[df["hour_utc"].dt.hour < 6].copy()

    df["city"] = city

    extra_weather.append(df)

extra_weather = pd.concat(
    extra_weather, ignore_index=True
)

print("Additional records:", len(extra_weather))
print("Cities:", extra_weather["city"].nunique())
print("Missing values:")
print(extra_weather.isna().sum())

Additional records: 48
Cities: 8
Missing values:
time                    0
temperature_2m          0
relative_humidity_2m    0
apparent_temperature    0
hour_utc                0
city                    0
dtype: int64


In [19]:
extra_weather = extra_weather[
    [
        "time",
        "temperature_2m",
        "relative_humidity_2m",
        "apparent_temperature",
        "city",
        "hour_utc"
    ]
]

weather = pd.concat(
    [weather, extra_weather],
    ignore_index=True
)

weather = weather.drop_duplicates(
    subset=["city", "hour_utc"]
).sort_values(["city", "hour_utc"])

print("Weather records:", len(weather))
print("Cities:", weather["city"].nunique())
print("Missing values:", weather.isna().sum().sum())

Weather records: 70128
Cities: 8
Missing values: 0


In [20]:
weather_summary = (
    weather.groupby("hour_utc", as_index=False)
    .agg(
        avg_temperature_c=("temperature_2m", "mean"),
        avg_humidity_pct=("relative_humidity_2m", "mean"),
        avg_feels_like_c=("apparent_temperature", "mean")
    )
)

electricity = electricity.copy()

electricity["weather_hour_utc"] = (
    electricity["hour_ending_utc"]
    - pd.Timedelta(hours=1)
)

integrated = electricity.merge(
    weather_summary,
    left_on="weather_hour_utc",
    right_on="hour_utc",
    how="left",
    validate="one_to_one"
)

print("Integrated shape:", integrated.shape)
print(
    "Unmatched records:",
    integrated["avg_temperature_c"].isna().sum()
)
print(
    "Duplicate electricity hours:",
    integrated["hour_ending_utc"].duplicated().sum()
)

Integrated shape: (8760, 11)
Unmatched records: 0
Duplicate electricity hours: 0


In [21]:
from pathlib import Path

output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

assert len(integrated) == 8760
assert integrated["avg_temperature_c"].notna().all()
assert integrated["avg_humidity_pct"].notna().all()
assert integrated["avg_feels_like_c"].notna().all()

integrated.to_csv(
    output_dir / "electricity_weather_integrated_2025.csv",
    index=False
)

weather.to_csv(
    output_dir / "weather_2025_extended.csv",
    index=False
)

print("Both datasets saved successfully!")

Both datasets saved successfully!
